In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.ensemble import RandomForestRegressor

import warnings
warnings.filterwarnings('ignore')

# Load Dataset
df = pd.read_excel(r"C:\Users\ak32a\Downloads\Anal-ysts\LTF Challenge data with dictionary.xlsx", sheet_name="TrainData")

# Normalize column names
df.columns = df.columns.str.strip().str.upper()

# Convert Ownership to lowercase
df['OWNERSHIP'] = df['OWNERSHIP'].astype(str).str.lower()

# Filter for parental ownership
parental_df = df[df['OWNERSHIP'] == 'parental']

# Drop rows with missing location info
parental_df = parental_df.dropna(subset=['CITY', 'VILLAGE', 'DISTRICT', 'STATE', 'TOTAL_LAND_FOR_AGRICULTURE'])

# Group and filter for repeated common location + land size
common_groups = parental_df.groupby(['CITY', 'VILLAGE', 'DISTRICT', 'STATE', 'TOTAL_LAND_FOR_AGRICULTURE']).size().reset_index(name='COUNT')
common_groups = common_groups[common_groups['COUNT'] > 1]

# Merge back to get full rows
result_df = parental_df.merge(common_groups, on=['CITY', 'VILLAGE', 'DISTRICT', 'STATE', 'TOTAL_LAND_FOR_AGRICULTURE'], how='inner')

# Result stats and save
print("Number of such rows:", len(result_df))
print(result_df.head())

# Save
result_df.to_csv("filtered_common_location_parental.csv", index=False)


Number of such rows: 362
           FARMERID           STATE   REGION SEX     CITY  ZIPCODE DISTRICT  \
0  1068465784357340          ODISHA     EAST   M  RAMGARH   754022  CUTTACK   
1  1574797854092240  MADHYA PRADESH  CENTRAL   M   KANNOD   455459    DEWAS   
2  2230326140558730  MADHYA PRADESH  CENTRAL   M   KANNOD   455440    DEWAS   
3  2885192446675250       JHARKHAND     EAST   M    MURHU   835216   KHUNTI   
4  3824389016926960  MADHYA PRADESH  CENTRAL   M  BERASIA   463106   BHOPAL   

      VILLAGE MARITAL_STATUS               LOCATION  ...  \
0     Karanji              S   20.649898,85.9164129  ...   
1  Golaguthan              M  22.5901019,76.9216425  ...   
2    Surmanya              M  22.4967458,76.4739731  ...   
3     Bamhani              M   23.0907831,85.279463  ...   
4        Agra              S   23.6160729,77.429152  ...   

  RABI SEASONS SEASONAL AVERAGE GROUNDWATER THICKNESS (CM) IN 2020  \
0                                              91.92                 

In [3]:
my_list = ['ko22_village_score_based_on_socio_economic_parameters_0_to_100', 'village_score_based_on_socio_economic_parameters_0_to_100',
 'k022_village_category_based_on_socio_economic_parameters_good,_average,_poor vs village_category_based_on_socio_economic_parameters_good,_average,_poor',
 'kharif_seasons__type_of_soil_in_2022', 'rabi_seasons_type_of_soil_in_2022',
 'kharif_seasons__type_of_soil_in_2022', 'rabi_seasons_type_of_soil_in_2021',
 'kharif_seasons__type_of_soil_in_2022', 'kharif_seasons_type_of_soil_in_2021',
 'kharif_seasons__type_of_soil_in_2022', 'rabi_seasons_type_of_soil_in_2020',
 'kharif_seasons__type_of_water_bodies_in_hectares_2022','rabi_seasons_type_of_water_bodies_in_hectares_2022',
 'kharif_seasons__type_of_water_bodies_in_hectares_2022','rabi_seasons_type_of_water_bodies_in_hectares_2021',
 'kharif_seasons__type_of_water_bodies_in_hectares_2022','kharif_seasons_type_of_water_bodies_in_hectares_2021',
 'kharif_seasons__type_of_water_bodies_in_hectares_2022','rabi_seasons_type_of_water_bodies_in_hectares_2020',
 'kharif_seasons__agro_ecological_sub_zone_in_2022','rabi_seasons_agro_ecological_sub_zone_in_2022',
 'kharif_seasons__agro_ecological_sub_zone_in_2022','rabi_seasons_agro_ecological_sub_zone_in_2021',
 'kharif_seasons__agro_ecological_sub_zone_in_2022','kharif_seasons_agro_ecological_sub_zone_in_2021',
 'kharif_seasons__agro_ecological_sub_zone_in_2022','rabi_seasons_agro_ecological_sub_zone_in_2020',
 'rabi_seasons__season_irrigated_area_in_2022','kharif_seasons_kharif_season_irrigated_area_in_2021',
 'rabi_seasons_cropping_density_in_2022','kharif_seasons_cropping_density_in_2021',
 'rabi_seasons_type_of_soil_in_2022','rabi_seasons_type_of_soil_in_2021',
 'rabi_seasons_type_of_soil_in_2022','kharif_seasons_type_of_soil_in_2021',
 'rabi_seasons_type_of_soil_in_2022','rabi_seasons_type_of_soil_in_2020',
 'rabi_seasons_type_of_water_bodies_in_hectares_2022','rabi_seasons_type_of_water_bodies_in_hectares_2021',
 'rabi_seasons_type_of_water_bodies_in_hectares_2022','kharif_seasons_type_of_water_bodies_in_hectares_2021',
 'rabi_seasons_type_of_water_bodies_in_hectares_2022','rabi_seasons_type_of_water_bodies_in_hectares_2020',
 'rabi_seasons_agro_ecological_sub_zone_in_2022','rabi_seasons_agro_ecological_sub_zone_in_2021',
 'rabi_seasons_agro_ecological_sub_zone_in_2022','kharif_seasons_agro_ecological_sub_zone_in_2021',
 'rabi_seasons_agro_ecological_sub_zone_in_2022','rabi_seasons_agro_ecological_sub_zone_in_2020',
 'rabi_seasons_type_of_soil_in_2021','kharif_seasons_type_of_soil_in_2021',
 'rabi_seasons_type_of_soil_in_2021','rabi_seasons_type_of_soil_in_2020',
 'rabi_seasons_type_of_water_bodies_in_hectares_2021','kharif_seasons_type_of_water_bodies_in_hectares_2021',
 'rabi_seasons_type_of_water_bodies_in_hectares_2021','rabi_seasons_type_of_water_bodies_in_hectares_2020',
 'rabi_seasons_agro_ecological_sub_zone_in_2021','kharif_seasons_agro_ecological_sub_zone_in_2021',
 'rabi_seasons_agro_ecological_sub_zone_in_2021','rabi_seasons_agro_ecological_sub_zone_in_2020',
 'kharif_seasons_type_of_soil_in_2021','rabi_seasons_type_of_soil_in_2020',
 'kharif_seasons_type_of_water_bodies_in_hectares_2021','rabi_seasons_type_of_water_bodies_in_hectares_2020',
 'kharif_seasons_agro_ecological_sub_zone_in_2021','rabi_seasons_agro_ecological_sub_zone_in_2020']


In [5]:
unique_values = list(set(my_list))

In [7]:
print(unique_values)

['kharif_seasons_cropping_density_in_2021', 'village_score_based_on_socio_economic_parameters_0_to_100', 'kharif_seasons__agro_ecological_sub_zone_in_2022', 'k022_village_category_based_on_socio_economic_parameters_good,_average,_poor vs village_category_based_on_socio_economic_parameters_good,_average,_poor', 'rabi_seasons_type_of_soil_in_2021', 'rabi_seasons_cropping_density_in_2022', 'kharif_seasons__type_of_soil_in_2022', 'rabi_seasons_agro_ecological_sub_zone_in_2020', 'rabi_seasons_type_of_soil_in_2020', 'rabi_seasons_type_of_water_bodies_in_hectares_2021', 'rabi_seasons_type_of_water_bodies_in_hectares_2022', 'rabi_seasons_agro_ecological_sub_zone_in_2021', 'rabi_seasons_type_of_water_bodies_in_hectares_2020', 'kharif_seasons_kharif_season_irrigated_area_in_2021', 'kharif_seasons_type_of_water_bodies_in_hectares_2021', 'kharif_seasons_agro_ecological_sub_zone_in_2021', 'rabi_seasons_type_of_soil_in_2022', 'ko22_village_score_based_on_socio_economic_parameters_0_to_100', 'kharif_